In [24]:
import os
import numpy as np
np.set_printoptions(legacy='1.25')

from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

from pymatgen.core import Structure
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer

import logging

logger = logging.getLogger("interface_finder")
logger.setLevel(logging.INFO)


# =========================
# CONFIGURATION
# =========================

@dataclass
class InterfaceConfig:
    film_max_miller: int = 4 #maximum value of h,k,l index for seeking surface orientation
    misfit_limit: float = 5.0 #maximum inconvenience between lattice sizes of oriented substrate and film in % 
    num_sites_limit: int = 200 #maximum number of atoms in slab with constructed interface
    density_limit: float = 0.1 #minimum number of surface atoms per substrate/film surface area to avoid voids in the inteface area 
    charge_limit: float = 0.1 #maximum absolute charge per interface area in the inteface layer (sum for substrate and film). Close to 0 is better
    gap: float = 2.0 #distance between substrate and film in A
    vacuum_over_film: float = 15.0 #width of vacuum layer. put ~2 A to get slab without vacuum
    film_thickness: float = 5 #minimum film thickness in A
    substrate_thickness: float = 7 #minimum substrate thickness in A
    surface_thickness: float = 0.8 #thickness of layer (x (sub suf)-interface-x(film suf)) considering as surface to calculate interface density and charge
    output_folder: str = "interfaces" #where to save interface structures


# =========================
# MAIN CLASS
# =========================

class InterfaceFinder:
    """
    Build and filter coherent interfaces between substrate and film.
    """

    def __init__(self, substrate: Structure, film: Structure, config: InterfaceConfig):
        self.substrate = substrate
        self.film = film
        self.cfg = config

        self.sub_formula = substrate.composition.reduced_formula
        self.film_formula = film.composition.reduced_formula

    # ---------- MATCHING ----------

    def generate_millers(max_index: int) -> List[Tuple[int, int, int]]:
        """Generate all Miller indices up to max_index excluding (0,0,0)."""
        millers = []
        for h, k, l in itertools.product(range(0, max_index + 1), repeat=3):
            if (h, k, l) != (0, 0, 0):
                millers.append((h, k, l))
        return millers
    
    
    def find_matches(
            substrate_bulk,
            film_bulk,
            mode: str = "grid",
            max_sub_miller: int = 2,
            max_film_miller: int = 2,
            sub_miller: Tuple[int, int, int] | None = None,
            film_miller: Tuple[int, int, int] | None = None,
            explicit_pairs: List[Tuple[Tuple[int,int,int], Tuple[int,int,int]]] | None = None,
            misfit_limit: float = 5.0,
        ) -> Dict[str, List[Dict[str, Any]]]:
            """
            Find epitaxial matches between substrate and film.
        
            Returns dict:
                {
                    "all": [...],        # all tested matches
                    "filtered": [...]    # matches satisfying misfit criterion
                }
            """
        
            results_all = []
            results_filtered = []
        
            # -------- define search space --------
        
            if mode == "grid":
                sub_millers = generate_millers(max_sub_miller)
                film_millers = generate_millers(max_film_miller)
                pairs = list(itertools.product(sub_millers, film_millers))
        
            elif mode == "fix_sub":
                if fixed_substrate is None:
                    raise ValueError("fixed_substrate must be provided in mode='fix_sub'")
                film_millers = generate_millers(max_film_miller)
                pairs = [(sub_miller, fm) for fm in film_millers]
        
            elif mode == "fix_film":
                if fixed_film is None:
                    raise ValueError("fixed_film must be provided in mode='fix_film'")
                sub_millers = generate_millers(max_sub_miller)
                pairs = [(sm, film_miller) for sm in sub_millers]
        
            elif mode == "explicit":
                if explicit_pairs is None:
                    raise ValueError("explicit_pairs must be provided in mode='explicit'")
                pairs = explicit_pairs
        
            else:
                raise ValueError(f"Unknown mode: {mode}")
        
            logger.info(f"Search mode: {mode}, number of pairs: {len(pairs)}")
        
            # -------- run pymatgen matching --------
        
            analyzer = SubstrateAnalyzer(
                film_max_miller=max_film_miller,
                bidirectional=False,
            )
        
            for sub_hkl, film_hkl in pairs:
        
                matches = analyzer.calculate(
                    film=film_bulk,
                    substrate=substrate_bulk,
                    substrate_millers=[sub_hkl],
                )
        
                for match in matches:
        
                    if match.film_miller != film_hkl:
                        continue
        
                    # substrate misfit
                    sub_vec = np.array(substrate_bulk.lattice.matrix[:2])
                    new_sub_vec = np.dot(match.substrate_transformation, sub_vec)
                    misfit_x_s = abs((np.linalg.norm(new_sub_vec[0]) - np.linalg.norm(sub_vec[0])) / np.linalg.norm(sub_vec[0])) * 100
                    misfit_y_s = abs((np.linalg.norm(new_sub_vec[1]) - np.linalg.norm(sub_vec[1])) / np.linalg.norm(sub_vec[1])) * 100
        
                    # film misfit
                    film_vec = np.array(film_bulk.lattice.matrix[:2])
                    new_film_vec = np.dot(match.film_transformation, film_vec)
                    misfit_x = abs((np.linalg.norm(new_film_vec[0]) - np.linalg.norm(film_vec[0])) / np.linalg.norm(film_vec[0])) * 100
                    misfit_y = abs((np.linalg.norm(new_film_vec[1]) - np.linalg.norm(film_vec[1])) / np.linalg.norm(film_vec[1])) * 100
        
                    # rounding
                    misfit_x_r = round(misfit_x, 1)
                    misfit_y_r = round(misfit_y, 1)
                    misfit_x_s_r = round(misfit_x_s, 1)
                    misfit_y_s_r = round(misfit_y_s, 1)
        
                    logger.info(
                        f"hkl(sub/film) = {sub_hkl}/{film_hkl} | "
                        f"misfit film = ({misfit_x_r}, {misfit_y_r}) %, "
                        f"substrate = ({misfit_x_s_r}, {misfit_y_s_r}) %"
                    )
        
                    record = dict(
                        hkl_sub=sub_hkl,
                        hkl_film=film_hkl,
                        misfit_film=(misfit_x_r, misfit_y_r),
                        misfit_sub=(misfit_x_s_r, misfit_y_s_r),
                        von_mises=round(match.von_mises_strain, 4),
                        match_obj=match,
                    )
        
                    results_all.append(record)
        
                    if misfit_x <= misfit_limit and misfit_y <= misfit_limit:
                        results_filtered.append(record)
        
            return {
                "all": results_all,
                "filtered": results_filtered,
            }

    # ---------- SURFACE PROPERTIES ----------

    def _select_surface_atoms(self, structure, select="top", thickness=1.0):
        z = np.array([s.coords[2] for s in structure])
        z_max, z_min = z.max(), z.min()

        if select == "top":
            return [s for s in structure if z_max - s.coords[2] <= thickness]
        elif select == "bottom":
            return [s for s in structure if s.coords[2] - z_min <= thickness]
        else:
            raise ValueError("select must be 'top' or 'bottom'")

    def compute_surface_density(self, structure, select="top"):
        a, b = structure.lattice.matrix[:2]
        area = np.linalg.norm(np.cross(a, b))
        atoms = self._select_surface_atoms(structure, select, self.cfg.surface_thickness)
        return len(atoms) / area

    def compute_surface_charge_density(self, structure, select="top"):
        a, b = structure.lattice.matrix[:2]
        area = np.linalg.norm(np.cross(a, b))
        atoms = self._select_surface_atoms(structure, select, self.cfg.surface_thickness)

        try:
            charge = sum(s.specie.oxi_state for s in atoms)
        except AttributeError:
            raise ValueError("Oxidation states must be assigned to structure.")

        return charge / area

    # ---------- INTERFACE GENERATION ----------

    def build_interfaces(self, substrate_miller, film_miller, match_info) -> List[Dict[str, Any]]:
        results = []

        zsl = ZSLGenerator(
            max_area=400,
            max_area_ratio_tol=0.05,
            max_length_tol=0.05,
            max_angle_tol=1,
            bidirectional=False,
        )

        cib = CoherentInterfaceBuilder(
            film_structure=self.film,
            substrate_structure=self.substrate,
            film_miller=film_miller,
            substrate_miller=substrate_miller,
            zslgen=zsl,
            filter_out_sym_slabs=True,
        )

        seen = set()

        for termination in cib.terminations:
            interfaces = cib.get_interfaces(
                termination=termination,
                gap=self.cfg.gap,
                vacuum_over_film=self.cfg.vacuum_over_film,
                film_thickness=self.cfg.film_thickness,
                substrate_thickness=self.cfg.substrate_thickness,
                in_layers=False,
            )

            for iface in interfaces:
                if iface.num_sites > self.cfg.num_sites_limit:
                    continue
                if termination in seen:
                    continue
                seen.add(termination)

                sub_dens = self.compute_surface_density(iface.substrate, "top")
                film_dens = self.compute_surface_density(iface.film, "bottom")

                sub_charge = self.compute_surface_charge_density(iface.substrate, "top")
                film_charge = self.compute_surface_charge_density(iface.film, "bottom")
                total_charge = abs(sub_charge + film_charge)

                if not (
                    sub_dens > self.cfg.density_limit
                    and film_dens > self.cfg.density_limit
                    and total_charge < self.cfg.charge_limit
                ):
                    continue

                t1 = termination[0].replace("/", "")
                t2 = termination[1].replace("/", "")
                fname = f"{self.sub_formula}_{self.film_formula}_{''.join(map(str,substrate_miller))}_{''.join(map(str,film_miller))}_{iface.num_sites}at_{t1}_{t2}"

                os.makedirs(f"{self.cfg.output_folder}/{self.sub_formula}", exist_ok=True)
                iface.to(filename=f"{self.cfg.output_folder}/{self.sub_formula}/{fname}.POSCAR")

                results.append(dict(
                    substrate=self.sub_formula,
                    film=self.film_formula,
                    hkl_sub=substrate_miller,
                    hkl_film=film_miller,
                    misfit_x=match_info[2][0],
                    misfit_y=match_info[2][1],
                    misfit_x_sub=match_info[4][0],
                    misfit_y_sub=match_info[4][1],
                    von_mises=match_info[3],
                    termination=termination,
                    n_at=iface.num_sites,
                    slab=fname,
                    substrate_density=sub_dens,
                    film_density=film_dens,
                    substrate_charge_density=sub_charge,
                    film_charge_density=film_charge,
                    abs_charge_density=total_charge,
                ))

        return results

    # ---------- FULL PIPELINE ----------

    def run(self, substrate_millers: List[Tuple[int, int, int]]) -> List[Dict[str, Any]]:
        all_results = []

        for hkl in substrate_millers:
            matches = self.find_matches(hkl)
            for m in matches:
                res = self.build_interfaces(m[0], m[1], m)
                all_results.extend(res)

        return all_results

In [26]:
substrate = Structure.from_file("Li3GaN2.cif")
substrate.add_oxidation_state_by_guess()

film = Structure.from_file("Li.cif")
film.add_oxidation_state_by_element({"Li": +1})

config = InterfaceConfig(
    misfit_limit=5.0,
    num_sites_limit=300,
    film_max_miller = 1,
    output_folder="interfaces_with_Li_mod"
)

finder = InterfaceFinder(substrate, film, config)

# interfaces = finder.run(substrate_millers=[(0,0,1),(1,1,0),(1,1,1)])
matches = finder.find_matches(substrate, film, sub_miller=[1,1,0])
# print(matches)


# import pandas as pd
# df = pd.DataFrame(interfaces)
# df.head()

ValueError: Unknown mode: Full Formula (Li2)
Reduced Formula: Li
abc   :   3.439312   3.439312   3.439312
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (2)
  #  SP      a    b    c
---  ----  ---  ---  ---
  0  Li+   0    0    0
  1  Li+   0.5  0.5  0.5

In [16]:
import pandas as pd
df = pd.DataFrame(matches)
# df.head()
df


,0,1,2,3,4
0,"[0, 0, 1]","(1, 0, 0)","[1.8284271247461905, 3.0000000000000004]",0.003583,"[0.0, 0.0]"


In [29]:
import itertools
import numpy as np
import logging
from typing import List, Tuple, Dict, Any
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer


logger = logging.getLogger("interface_finder")
logger.setLevel(logging.INFO)


def generate_millers(max_index: int) -> List[Tuple[int, int, int]]:
    """Generate all Miller indices up to max_index excluding (0,0,0)."""
    millers = []
    for h, k, l in itertools.product(range(0, max_index + 1), repeat=3):
        if (h, k, l) != (0, 0, 0):
            millers.append((h, k, l))
    return millers


def find_matches(
    substrate_bulk,
    film_bulk,
    mode: str = "grid",
    max_sub_miller: int = 2,
    max_film_miller: int = 2,
    fixed_substrate: Tuple[int, int, int] | None = None,
    fixed_film: Tuple[int, int, int] | None = None,
    explicit_pairs: List[Tuple[Tuple[int,int,int], Tuple[int,int,int]]] | None = None,
    misfit_limit: float = 5.0,
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Find epitaxial matches between substrate and film.

    Returns dict:
        {
            "all": [...],        # all tested matches
            "filtered": [...]    # matches satisfying misfit criterion
        }
    """

    results_all = []
    results_filtered = []

    # -------- define search space --------

    if mode == "grid":
        sub_millers = generate_millers(max_sub_miller)
        film_millers = generate_millers(max_film_miller)
        pairs = list(itertools.product(sub_millers, film_millers))

    elif mode == "fix_sub":
        if fixed_substrate is None:
            raise ValueError("fixed_substrate must be provided in mode='fix_sub'")
        film_millers = generate_millers(max_film_miller)
        pairs = [(fixed_substrate, fm) for fm in film_millers]

    elif mode == "fix_film":
        if fixed_film is None:
            raise ValueError("fixed_film must be provided in mode='fix_film'")
        sub_millers = generate_millers(max_sub_miller)
        pairs = [(sm, fixed_film) for sm in sub_millers]

    elif mode == "explicit":
        if explicit_pairs is None:
            raise ValueError("explicit_pairs must be provided in mode='explicit'")
        pairs = explicit_pairs

    else:
        raise ValueError(f"Unknown mode: {mode}")

    logger.info(f"Search mode: {mode}, number of pairs: {len(pairs)}")

    # -------- run pymatgen matching --------

    analyzer = SubstrateAnalyzer(
        film_max_miller=max_film_miller,
        bidirectional=False,
    )

    for sub_hkl, film_hkl in pairs:

        matches = analyzer.calculate(
            film=film_bulk,
            substrate=substrate_bulk,
            substrate_millers=[sub_hkl],
        )

        for match in matches:

            if match.film_miller != film_hkl:
                continue

            # substrate misfit
            sub_vec = np.array(substrate_bulk.lattice.matrix[:2])
            new_sub_vec = np.dot(match.substrate_transformation, sub_vec)
            misfit_x_s = abs((np.linalg.norm(new_sub_vec[0]) - np.linalg.norm(sub_vec[0])) / np.linalg.norm(sub_vec[0])) * 100
            misfit_y_s = abs((np.linalg.norm(new_sub_vec[1]) - np.linalg.norm(sub_vec[1])) / np.linalg.norm(sub_vec[1])) * 100

            # film misfit
            film_vec = np.array(film_bulk.lattice.matrix[:2])
            new_film_vec = np.dot(match.film_transformation, film_vec)
            misfit_x = abs((np.linalg.norm(new_film_vec[0]) - np.linalg.norm(film_vec[0])) / np.linalg.norm(film_vec[0])) * 100
            misfit_y = abs((np.linalg.norm(new_film_vec[1]) - np.linalg.norm(film_vec[1])) / np.linalg.norm(film_vec[1])) * 100

            # rounding
            misfit_x_r = round(misfit_x, 1)
            misfit_y_r = round(misfit_y, 1)
            misfit_x_s_r = round(misfit_x_s, 1)
            misfit_y_s_r = round(misfit_y_s, 1)

            logger.info(
                f"hkl(sub/film) = {sub_hkl}/{film_hkl} | "
                f"misfit film = ({misfit_x_r}, {misfit_y_r}) %, "
                f"substrate = ({misfit_x_s_r}, {misfit_y_s_r}) %"
            )

            record = dict(
                hkl_sub=sub_hkl,
                hkl_film=film_hkl,
                misfit_film=(misfit_x_r, misfit_y_r),
                misfit_sub=(misfit_x_s_r, misfit_y_s_r),
                von_mises=round(match.von_mises_strain, 4),
                match_obj=match,
            )

            results_all.append(record)

            if misfit_x <= misfit_limit and misfit_y <= misfit_limit:
                results_filtered.append(record)

    return {
        "all": results_all,
        "filtered": results_filtered,
    }

In [31]:
substrate_bulk = Structure.from_file("Li3GaN2.cif")
# substrate_bulk.add_oxidation_state_by_guess()

film_bulk = Structure.from_file("Li.cif")
# film_bulk.add_oxidation_state_by_element({"Li": +1})

In [39]:
res = find_matches(
    substrate_bulk,
    film_bulk,
    mode="explicit",
    explicit_pairs = [[[1,1,0],[1,1,0]]],
    # fixed_film = [1,1,0],
    misfit_limit=5.0,
)

all_matches = res["all"]
good_matches = res["filtered"]

In [41]:
res

{'all': [], 'filtered': []}